# dataloader-pin-memory-workers — faded example 2: Oversample a minority class with WeightedRandomSampler

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataloader-pin-memory-workers`. Running the beacon reports progress on the `PyTorch: DataLoader pin_memory + workers` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: DataLoader pin_memory + workers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataloader-pin-memory-workers`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataloader-pin-memory-workers"
DD_SUBTOPIC = "PyTorch: DataLoader pin_memory + workers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

`WeightedRandomSampler(weights, num_samples, replacement=True)` draws indices in proportion to per-sample `weights`, so rare classes can be oversampled to balance an imbalanced dataset. Because it provides a sampler, you must NOT also pass `shuffle=True` (the two are mutually exclusive).

## Faded exercise 2

### Balance an imbalanced dataset by oversampling

You have 100 items: indices 0..89 are class 0 and 90..99 are class 1 (a 9:1 imbalance). Implement `make_balanced_loader(labels, batch_size)` that returns a `DataLoader`-ready `WeightedRandomSampler` setup where each *class* is equally likely to be drawn. The per-sample weight is `1 / (count of that sample's class)`. Draw `num_samples=len(labels)` with replacement.

Complete the blanked line that builds the per-sample weight tensor.

**Fill in:** Builds the per-sample weight tensor as the inverse of each sample's class frequency.

In [ ]:
from torch.utils.data import WeightedRandomSampler

def make_balanced_sampler(labels):
    labels = t.as_tensor(labels)
    class_count = t.bincount(labels)
    sample_weights = 1.0 / class_count[labels].float()
    sampler = WeightedRandomSampler(
        weights=sample_weights.double(),
        num_samples=len(labels),
        replacement=True,
    )
    return sampler


def _test():
    t.manual_seed(0)
    labels = t.cat([t.zeros(90, dtype=t.long), t.ones(10, dtype=t.long)])
    sampler = make_balanced_sampler(labels)
    drawn = labels[list(sampler)]
    n1 = int((drawn == 1).sum())
    n0 = int((drawn == 0).sum())
    assert n0 + n1 == len(labels), (n0, n1)
    # with inverse-frequency weights each class should be roughly half
    frac1 = n1 / len(labels)
    assert 0.35 < frac1 < 0.65, frac1
    # independent weight check
    cc = t.bincount(labels)
    w = 1.0 / cc[labels].float()
    assert t.allclose(w[labels == 0][0], t.tensor(1.0 / 90.0))
    assert t.allclose(w[labels == 1][0], t.tensor(1.0 / 10.0))


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
from torch.utils.data import WeightedRandomSampler

def make_balanced_sampler(labels):
    labels = t.as_tensor(labels)
    class_count = t.bincount(labels)
    sample_weights = 1.0 / class_count[labels].float()
    sampler = WeightedRandomSampler(
        weights=sample_weights.double(),
        num_samples=len(labels),
        replacement=True,
    )
    return sampler
```
</details>